# **multimoda-rs**: Intravascular Module – Notebook Tutorial

This notebook follows the [Intravascular Alignment Tutorial](../tutorial_intravascular.rst) step-by-step:

1. General note on segmentation preparation
2. Workflow from CSV files (AIVUS-CAA format)
3. Workflow from numpy arrays — including parameter fine-tuning
4. Alignment with a CCTA centerline
5. Saving geometries as `.obj` files
6. Utility functions: `to_array` and `numpy_to_geometry`
7. Class-level methods for `PyContour`, `PyFrame`, and `PyGeometry`

> **Performance note:** Alignment is compute-intensive. Running the notebook via the console is faster than interactive cell-by-cell execution.

## 1. General note on segmentation preparation

The package works with segmented intravascular images from IVUS or OCT.
Segmentation must follow specific conventions for optimal results:

- Files follow AIVUS-CAA convention: `diastolic_contours.csv`, `systolic_contours.csv`,
  `diastolic_reference_points.csv`, `systolic_reference_points.csv`.
- Each file has columns `(frame_index, x_mm, y_mm, z_mm)` — **no header row**.
- A correct anatomical **reference point** is essential for downstream centerline co-registration.

For CAD a bifurcation serves as reference; for AAOCA the reference is placed at the aortic side of
the most proximal fully closed lumen contour (see tutorial for details).

**Four processing modes**

| Mode | Inputs | Typical use-case |
|------|--------|-----------------|
| `full` | 4 states (rest-dia, rest-sys, stress-dia, stress-sys) | AAOCA pulsatile lumen analysis |
| `doublepair` | 2 independent pullbacks, each with dia+sys | Pre/post-stenting across both phases |
| `singlepair` | 1 pullback with dia+sys | CAD: diastole vs. systole within one pullback |
| `single` | 1 pullback (dia or sys only) | Standalone reconstruction (CAD, OCT) |

In [1]:
import os
from pathlib import Path
import numpy as np
import multimodars as mm

cwd = Path.cwd()
for candidate in [cwd, cwd.parent, cwd.parent.parent]:
    if (candidate / "examples" / "data").exists():
        os.chdir(candidate / "examples" / "data")
        break
    elif (candidate / "data").exists():
        os.chdir(candidate / "data")
        break
print(f"Working directory: {os.getcwd()}")

Working directory: /mnt/d/00_coding/multimoda-rs/examples/data


In [2]:
%pip --disable-pip-version-check install trimesh plotly scipy nbformat | grep -v 'already satisfied'


import trimesh
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"
from plotly.subplots import make_subplots


def trimesh_to_mesh3d(mesh, color, name, opacity=0.6):
    verts = mesh.vertices
    faces = mesh.faces
    return go.Mesh3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        color=color, opacity=opacity, name=name, flatshading=True,
    )


def plot_pair(before_paths, after_paths, colors, titles):
    """Side-by-side 3D comparison of two mesh lists."""
    before_meshes = [trimesh.load(p) for p in before_paths]
    after_meshes = [trimesh.load(p) for p in after_paths]
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{"type": "scene"}, {"type": "scene"}]],
        subplot_titles=titles,
    )
    labels = ["diastole", "systole"]
    for mesh, color, label in zip(before_meshes, colors, labels):
        fig.add_trace(trimesh_to_mesh3d(mesh, color, f"before_{label}"), row=1, col=1)
    for mesh, color, label in zip(after_meshes, colors, labels):
        fig.add_trace(trimesh_to_mesh3d(mesh, color, f"after_{label}"), row=1, col=2)
    camera = dict(eye=dict(x=1.5, y=1.5, z=1.0))
    fig.update_layout(
        width=900, height=450,
        scene_camera=camera, scene2_camera=camera,
        scene=dict(aspectmode="data"), scene2=dict(aspectmode="data"),
        margin=dict(l=0, r=0, t=30, b=0),
    )
    fig.show()

/mnt/d/00_coding/multimoda-rs/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


## 2. Workflow from CSV files

This section uses AIVUS-CAA CSV files.
`from_file_full` processes all four states simultaneously (rest-diastole, rest-systole,
stress-diastole, stress-systole) and writes aligned meshes to four output directories.

See the [tutorial](../tutorial_intravascular.rst) for a detailed description of the CSV format.

In [3]:
# Full 4-state workflow: rest vs. stress, diastole vs. systole
rest, stress, dia, sys, _ = mm.from_file_full(
    input_path_ab="ivus_rest",
    input_path_cd="ivus_stress",
    labels=["rest_dia", "rest_sys", "stress_dia", "stress_sys"],
    step_rotation_deg=0.1,
    range_rotation_deg=90,
    image_center=(4.5, 4.5),
    radius=0.5,
    n_points=20,
    write_obj=True,
    watertight=False,
    contour_types=[mm.PyContourType.Lumen, mm.PyContourType.Catheter, mm.PyContourType.Wall],
    output_path_ab="output/rest",
    output_path_cd="output/stress",
    output_path_ac="output/diastole",
    output_path_bd="output/systole",
    interpolation_steps=28,
)

process_directory: unknown mapping name 'catheter', skipping
eem file not found, skipping: "ivus_rest/eem_diastolic_contours.csv"
eem file not found, skipping: "ivus_rest/eem_systolic_contours.csv"
process_directory: unknown mapping name 'catheter', skipping
process_directory: unknown mapping name 'catheter', skipping
eem file not found, skipping: "ivus_stress/eem_diastolic_contours.csv"
eem file not found, skipping: "ivus_stress/eem_systolic_contours.csv"
process_directory: unknown mapping name 'catheter', skipping



✅ Successfully built geometry from path
-----------------------------------------
✅ Lumen
❌ Eem
✅ Catheter
❌ Structures (0)
-----------------------------------------
Label: rest_dia
Diastole phase: Yes


✅ Successfully built geometry from path
-----------------------------------------
✅ Lumen
❌ Eem
✅ Catheter
❌ Structures (0)
-----------------------------------------
Label: rest_sys
Diastole phase: No


✅ Successfully built geometry from path
-----------------------------------------
✅ Lumen
❌ Eem
✅ Catheter
❌ Structures (0)
-----------------------------------------
Label: stress_dia
Diastole phase: Yes


✅ Successfully built geometry from path
-----------------------------------------
✅ Lumen
❌ Eem
✅ Catheter
❌ Structures (0)
-----------------------------------------
Label: stress_sys
Diastole phase: No


+--------------------------------------------------------------------+
|          ✅ Finished aligning 'rest_sys' (anomalous: true)          |
+---------+------------+---------------

In [4]:
# Build "before" meshes from raw CSV, then write explicit .obj files for comparison
rest_dia_raw = np.genfromtxt("ivus_rest/diastolic_contours.csv")
rest_dia_ref = np.genfromtxt("ivus_rest/diastolic_reference_points.csv")
rest_sys_raw = np.genfromtxt("ivus_rest/systolic_contours.csv")
rest_sys_ref = np.genfromtxt("ivus_rest/systolic_reference_points.csv")

rest_dia_before = mm.numpy_to_geometry(
    lumen_arr=rest_dia_raw, eem_arr=np.array([]), catheter_arr=np.array([]),
    wall_arr=np.array([]), reference_arr=rest_dia_ref, label="dia_before",
)
rest_sys_before = mm.numpy_to_geometry(
    lumen_arr=rest_sys_raw, eem_arr=np.array([]), catheter_arr=np.array([]),
    wall_arr=np.array([]), reference_arr=rest_sys_ref, label="sys_before",
)
rest_dia_before.frames = np.array([f.sort_frame_points() for f in rest_dia_before.frames])
rest_sys_before.frames = np.array([f.sort_frame_points() for f in rest_sys_before.frames])

# Write before (unprocessed) and after (aligned) meshes with explicit names
mm.to_obj(rest_dia_before, "output/before", watertight=False,
          contour_types=[mm.PyContourType.Lumen], filename_prefix="dia")
mm.to_obj(rest_sys_before, "output/before", watertight=False,
          contour_types=[mm.PyContourType.Lumen], filename_prefix="sys")
mm.to_obj(rest.geom_a, "output/after", watertight=False,
          contour_types=[mm.PyContourType.Lumen], filename_prefix="dia")
mm.to_obj(rest.geom_b, "output/after", watertight=False,
          contour_types=[mm.PyContourType.Lumen], filename_prefix="sys")

Successfully wrote lumen to output/before/dia_lumen.obj
Successfully wrote lumen to output/before/sys_lumen.obj
Successfully wrote lumen to output/after/dia_lumen.obj
Successfully wrote lumen to output/after/sys_lumen.obj


In [5]:
plot_pair(
    before_paths=["output/before/dia_lumen.obj", "output/before/sys_lumen.obj"],
    after_paths=["output/after/dia_lumen.obj", "output/after/sys_lumen.obj"],
    colors=["royalblue", "firebrick"],
    titles=["Before Processing", "After Processing"],
)

The data is now neatly ordered in pairs (diastolic and systolic geometry). Every geometry contains
contours for lumen, wall, and a synthetic catheter. The reference point is used for centerline
co-registration. All points belonging to a contour are stored in a `PyContour` struct.

In [6]:
print(f"PyGeometryPair:\n{rest}")
print(f"PyGeometry:\n{rest.geom_a}")
print(f"PyFrame:\n{rest.geom_a.frames[0]}")
print(f"PyContour:\n{rest.geom_a.frames[0].lumen}")
print(f"Extras:\n{rest.geom_a.frames[0].extras}")
print(f"PyContourPoint:\n{rest.geom_a.frames[0].lumen.points[0]}")

PyGeometryPair:
GeometryPair rest_dia - rest_sys (diastolic: 20 frames, systolic: 17 frames)
PyGeometry:
Geometry(20 frames, label='rest_dia', master_reference=385)
PyFrame:
Frame(id=0, centroid=(3.72, 5.25, 0.00), lumen=Contour(id=0, frame=385, points=501, centroid=(3.72, 5.25, 0.00), kind=Lumen), extras=2, structures=0)
PyContour:
Contour(id=0, frame=385, points=501, centroid=(3.72, 5.25, 0.00), kind=Lumen)
Extras:
{'Catheter': Contour(id=0, frame=385, points=20, centroid=(4.50, 4.50, 0.00), kind=Catheter), 'Wall': Contour(id=0, frame=385, points=501, centroid=(3.72, 5.25, 0.00), kind=Wall)}
PyContourPoint:
Point(frame_id=19, pt_id=0, x=3.80, y=7.90, z=0.00, aortic=false)


Different stenosis measurements can be obtained directly from the objects:

In [7]:
# Summary over PyGeometryPair
summary_pair, deform_table = rest.get_summary()
print(f"PyGeometryPair summary (dia_geom: (mla [mm²], max. stenosis, stenosis length [mm])):\n{summary_pair}")
print(f"Deformation table:\n{np.array(deform_table)}")

# Summary over PyGeometry
print(f"\nPyGeometry summary:\n{rest.geom_a.get_summary()[0]}")

# Per-contour measurements
print(f"\nContour area [mm²]: {rest.geom_a.frames[0].lumen.get_area():.4f}")
print(f"Elliptic ratio:      {rest.geom_a.frames[-1].lumen.get_elliptic_ratio():.4f}")

PyGeometryPair summary (dia_geom: (mla [mm²], max. stenosis, stenosis length [mm])):
((5.559206007496853, 0.6876787384433911, 10.161599999999993), (5.591747962523866, 0.6733428441447118, 11.431799999999992))
Deformation table:
[[ 0.          5.63662776  4.58651358  5.66441584  4.94168797  0.        ]
 [ 1.          5.79662471  4.75220339  5.59174796  5.083395    1.2702    ]
 [ 2.          5.70440311  3.69759593  5.61881953  4.86541236  2.5404    ]
 [ 3.          5.55920601  2.56932308  6.04871689  4.39939618  3.8106    ]
 [ 4.          5.5883072   1.63583386  6.25067009  3.47634844  5.0808    ]
 [ 5.          6.41743783  1.50724622  6.49932984  2.85135996  6.351     ]
 [ 6.          7.25510456  1.36697234  6.83711181  2.20372055  7.6212    ]
 [ 7.          7.74398201  1.24613112  7.52092177  1.93099775  8.8914    ]
 [ 8.          7.77347285  1.43038361  7.58783996  1.69170198 10.1616    ]
 [ 9.          9.05387612  1.32700846  7.34662992  1.6330749  11.4318    ]
 [10.         11.915209

The four pairs represent all four possible comparisons in gated images — particularly relevant for
coronary artery anomalies (AAOCA): rest pulsatile, stress pulsatile, stress-induced diastolic, and
stress-induced systolic lumen deformation.

The `interpolation_steps` parameter generates intermediate meshes between paired states, useful for
deformation animations in Blender. Set `interpolation_steps=0` to skip interpolation.

### Coronary Artery Disease

`from_file_single` reconstructs a single pullback with all contour layers (lumen, EEM, wall, catheter):

In [8]:
cad, _ = mm.from_file_single(
    input_path="ivus_full",
    diastole=True,
    step_rotation_deg=0.1,
    range_rotation_deg=90,
    image_center=(4.5, 4.5),
    radius=0.5,
    n_points=20,
    write_obj=True,
    watertight=False,
    contour_types=[
        mm.PyContourType.Lumen, mm.PyContourType.Eem,
        mm.PyContourType.Catheter, mm.PyContourType.Wall,
    ],
    output_path="output/cad",
)
# Centre coordinate system on EEM and write both versions
cad_aligned = cad.center_to_contour(mm.PyContourType.Eem)
mm.to_obj(cad_aligned, "output/cad", watertight=False,
          contour_types=[mm.PyContourType.Lumen, mm.PyContourType.Eem, mm.PyContourType.Wall],
          filename_prefix="aligned")

# from_file_single writes files as {contour_type}_{input_path_name}.obj
lumen = trimesh.load("output/cad/lumen_ivus_full.obj")
eem   = trimesh.load("output/cad/eem_ivus_full.obj")
wall  = trimesh.load("output/cad/wall_ivus_full.obj")
lumen_a = trimesh.load("output/cad/aligned_lumen.obj")
eem_a   = trimesh.load("output/cad/aligned_eem.obj")
wall_a  = trimesh.load("output/cad/aligned_wall.obj")

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=("CAD (lumen-centred)", "CAD (EEM-centred)"),
)
camera = dict(eye=dict(x=1.5, y=1.5, z=1.0))
for t in [trimesh_to_mesh3d(lumen, "firebrick", "Lumen", 1.0),
          trimesh_to_mesh3d(eem,   "royalblue", "EEM",   0.6),
          trimesh_to_mesh3d(wall,  "white",     "Wall",  0.5)]:
    fig.add_trace(t, row=1, col=1)
for t in [trimesh_to_mesh3d(lumen_a, "firebrick", "Lumen", 1.0),
          trimesh_to_mesh3d(eem_a,   "royalblue", "EEM",   0.6),
          trimesh_to_mesh3d(wall_a,  "white",     "Wall",  0.5)]:
    fig.add_trace(t, row=1, col=2)
fig.update_layout(
    width=900, height=450,
    scene=dict(camera=camera, aspectmode="data"),
    scene2=dict(camera=camera, aspectmode="data"),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()


✅ Successfully built geometry from path
-----------------------------------------
✅ Lumen
✅ Eem
✅ Catheter
✅ Structures (26)
-----------------------------------------
Label: ivus_full
Diastole phase: Yes


+--------------------------------------------------------------------+
|         ✅ Finished aligning 'ivus_full' (anomalous: false)         |
+---------+------------+---------------+-------+-------+-------------+
| Contour | Matched To | Rotation (°) |  Tx   |  Ty   |  Centroid   |
+---------+------------+---------------+-------+-------+-------------+
| 1       | 0          | 0.00          | 0.04  | -0.07 | (4.38,4.06) |
| 2       | 1          | -0.60         | 0.11  | -0.15 | (4.38,4.06) |
| 3       | 2          | 19.20         | 0.04  | -0.24 | (4.38,4.06) |
| 4       | 3          | 90.00         | -0.12 | -0.46 | (4.38,4.06) |
| 5       | 4          | 5.30          | -0.24 | -0.36 | (4.38,4.06) |
| 6       | 5          | -36.00        | -0.16 | -0.44 | (4.38,4.06) |
| 7       | 6

process_directory: unknown mapping name 'catheter', skipping


### Pre- vs. Post-stenting

For pre/post-stenting comparison across both cardiac phases, `from_file_full` is used with the
pre-stent pullback as `input_path_ab` and the post-stent pullback as `input_path_cd`.

In [9]:
prestent, poststent, dia_comp, sys_comp, _ = mm.from_file_full(
    input_path_ab="ivus_prestent",
    input_path_cd="ivus_poststent",
    step_rotation_deg=0.1,
    range_rotation_deg=45,
    watertight=False,
    output_path_ab="output/stent_rest",
    output_path_cd="output/stent_stress",
    output_path_ac="output/stent_diastole",
    output_path_bd="output/stent_systole",
    interpolation_steps=0,
)
# Write comparison geometries with explicit names
mm.to_obj(dia_comp.geom_a, "output/stent_vis", watertight=False,
          contour_types=[mm.PyContourType.Lumen], filename_prefix="prestent")
mm.to_obj(dia_comp.geom_b, "output/stent_vis", watertight=False,
          contour_types=[mm.PyContourType.Lumen], filename_prefix="poststent")

mesh_pre  = trimesh.load("output/stent_vis/prestent_lumen.obj")
mesh_post = trimesh.load("output/stent_vis/poststent_lumen.obj")

fig = go.Figure(data=[
    trimesh_to_mesh3d(mesh_pre,  "royalblue", "Pre-stent (diastole)"),
    trimesh_to_mesh3d(mesh_post, "firebrick", "Post-stent (diastole)"),
])
fig.update_layout(
    title="Pre- vs. Post-stenting: diastolic lumen comparison",
    scene=dict(aspectmode="data"),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()

process_directory: unknown mapping name 'catheter', skipping
eem file not found, skipping: "ivus_prestent/eem_diastolic_contours.csv"
eem file not found, skipping: "ivus_prestent/eem_systolic_contours.csv"
process_directory: unknown mapping name 'catheter', skipping
eem file not found, skipping: "ivus_poststent/eem_diastolic_contours.csv"
process_directory: unknown mapping name 'catheter', skipping
eem file not found, skipping: "ivus_poststent/eem_systolic_contours.csv"
process_directory: unknown mapping name 'catheter', skipping



✅ Successfully built geometry from path
-----------------------------------------
✅ Lumen
❌ Eem
✅ Catheter
❌ Structures (0)
-----------------------------------------
Label: ivus_prestent
Diastole phase: Yes


✅ Successfully built geometry from path
-----------------------------------------
✅ Lumen
❌ Eem
✅ Catheter
❌ Structures (0)
-----------------------------------------
Label: ivus_prestent
Diastole phase: No


✅ Successfully built geometry from path
-----------------------------------------
✅ Lumen
❌ Eem
✅ Catheter
❌ Structures (0)
-----------------------------------------
Label: ivus_poststent
Diastole phase: Yes


✅ Successfully built geometry from path
-----------------------------------------
✅ Lumen
❌ Eem
✅ Catheter
❌ Structures (0)
-----------------------------------------
Label: ivus_poststent
Diastole phase: No


+--------------------------------------------------------------------+
|       ✅ Finished aligning 'ivus_prestent' (anomalous: true)        |
+---------+----------

## 3. Workflow from numpy arrays

The numpy workflow accepts data from **any source**: you build `PyInputData` objects manually and
then call the same alignment functions. Use this workflow when your segmentation software uses
different file formats, when you need pre-processing steps, or when embedding `multimodars` into a
larger pipeline.

For stent comparison, both pre- and post-stent geometries are packed into `PyInputData` objects:

In [10]:
before_arr = np.genfromtxt("ivus_prestent/diastolic_contours.csv", delimiter='\t')
before_ref = np.genfromtxt("ivus_prestent/diastolic_reference_points.csv", delimiter='\t')
after_arr  = np.genfromtxt("ivus_poststent/diastolic_contours.csv", delimiter='\t')
after_ref  = np.genfromtxt("ivus_poststent/diastolic_reference_points.csv", delimiter='\t')

before_input = mm.numpy_to_inputdata(
    lumen_arr=before_arr, reference_points=before_ref,
    record=None, diastole=True, label="prestent",
)
after_input = mm.numpy_to_inputdata(
    lumen_arr=after_arr, reference_points=after_ref,
    record=None, diastole=True, label="poststent",
)

pair, _ = mm.from_array_singlepair(
    input_data_a=before_input,
    input_data_b=after_input,
    step_rotation_deg=0.01,
    range_rotation_deg=30,
    output_path="output/stent_comparison",
)


✅ Successfully built geometry from input data
-----------------------------------------
✅ Lumen
❌ Eem
✅ Catheter
❌ Structures (0)
-----------------------------------------
Label: prestent
Diastole phase: Yes


✅ Successfully built geometry from input data
-----------------------------------------
✅ Lumen
❌ Eem
✅ Catheter
❌ Structures (0)
-----------------------------------------
Label: poststent
Diastole phase: Yes


+--------------------------------------------------------------------+
|          ✅ Finished aligning 'prestent' (anomalous: true)          |
+---------+------------+---------------+-------+-------+-------------+
| Contour | Matched To | Rotation (°) |  Tx   |  Ty   |  Centroid   |
+---------+------------+---------------+-------+-------+-------------+
| 1       | 0          | -19.43        | 0.23  | 0.35  | (4.32,5.28) |
| 2       | 1          | 2.77          | 0.08  | 0.17  | (4.32,5.28) |
| 3       | 2          | -8.61         | 0.14  | 0.69  | (4.32,5.28) |
| 4       |

A single 3D geometry can also be reconstructed from OCT. `from_array_single` returns a
`PyGeometry` from a single-state contour array. The `replace_frame` method demonstrates
in-place editing before writing:

In [11]:
oct_raw = np.genfromtxt("oct_single/oct_contours_raw.csv", delimiter=',')
oct_ref = np.genfromtxt("oct_single/oct_ref.csv", delimiter=',')

oct_input = mm.numpy_to_inputdata(
    lumen_arr=oct_raw, reference_points=oct_ref,
    record=None, diastole=True, label="oct",
)

oct_recon, _ = mm.from_array_single(
    input_data=oct_input,
    step_rotation_deg=0.01,
    range_rotation_deg=6,
    image_center=(5.0, 5.0),
    radius=0.5,
    n_points=40,
    write_obj=False,
    watertight=False,
    output_path="output/oct",
    smooth=False,
)

# Replace an outlier frame and write both versions for comparison
frame = oct_recon.get_frame_at_z(34.8)
oct_replaced = oct_recon.replace_frame(frame.id + 1, frame)

mm.to_obj(oct_recon,    "output/oct", watertight=False,
          contour_types=[mm.PyContourType.Lumen], filename_prefix="oct")
mm.to_obj(oct_replaced, "output/oct", watertight=False,
          contour_types=[mm.PyContourType.Lumen], filename_prefix="oct_replaced")

oct_mesh       = trimesh.load("output/oct/oct_lumen.obj")
oct_mesh_fixed = trimesh.load("output/oct/oct_replaced_lumen.obj")

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=["OCT original", "OCT with replaced frame"],
)
scene_cfg = dict(aspectmode="data", camera=dict(eye=dict(x=1.5, y=1.5, z=1.0)))
for t in [trimesh_to_mesh3d(oct_mesh, "royalblue", "OCT Lumen")]:
    fig.add_trace(t, row=1, col=1)
for t in [trimesh_to_mesh3d(oct_mesh_fixed, "tomato", "OCT Replaced")]:
    fig.add_trace(t, row=1, col=2)
fig.update_layout(
    width=1200, height=600,
    scene=scene_cfg, scene2=scene_cfg,
    margin=dict(l=0, r=0, t=40, b=0),
)
fig.show()


✅ Successfully built geometry from input data
-----------------------------------------
✅ Lumen
❌ Eem
✅ Catheter
❌ Structures (0)
-----------------------------------------
Label: oct
Diastole phase: Yes


+--------------------------------------------------------------------+
|            ✅ Finished aligning 'oct' (anomalous: false)            |
+---------+------------+---------------+-------+-------+-------------+
| Contour | Matched To | Rotation (°) |  Tx   |  Ty   |  Centroid   |
+---------+------------+---------------+-------+-------+-------------+
| 1       | 0          | -1.75         | 0.00  | 0.05  | (6.30,5.97) |
| 2       | 1          | 1.40          | 0.02  | 0.12  | (6.30,5.97) |
| 3       | 2          | 0.47          | -0.03 | 0.17  | (6.30,5.97) |
| 4       | 3          | 3.10          | -0.04 | 0.25  | (6.30,5.97) |
| 5       | 4          | -1.86         | -0.06 | 0.29  | (6.30,5.97) |
| 6       | 5          | 4.01          | -0.10 | 0.42  | (6.30,5.97) |
| 7       | 6 

### Alignment algorithm — parameter fine-tuning

`from_array_singlepair` (and all `from_array_*` variants) expose every alignment parameter.
The example below uses `from_array_singlepair` with a `record` file (AIVUS-CAA combined CSV):

**Key parameters:**

- `step_rotation_deg` / `range_rotation_deg` — angular search space; ±range at step resolution.
  Reducing `range_rotation_deg` when the orientation is roughly known cuts runtime substantially.
- `sample_size` — contours are downsampled to at most this many points before Hausdorff distance
  computation. Default 500 is appropriate for most IVUS datasets.
- `image_center`, `radius`, `n_points` — define the synthetic catheter used as a rotational anchor.
  Larger `n_points` weights the catheter more heavily; `n_points=0` disables the catheter.
- `bruteforce` — sweeps the full angular range without hierarchical refinement. Avoid for routine use.
- `smooth` — 3-point moving average over each contour after alignment. Recommended.
- `postprocessing` — equalises axial frame spacing across pullbacks. Recommended when heart rate
  differs between conditions (e.g. rest vs. stress).

In [12]:
record   = np.genfromtxt("ivus_rest/combined_sorted_manual.csv", delimiter=',', skip_header=1)
dia_cont = np.genfromtxt("ivus_rest/diastolic_contours.csv",       delimiter='\t')
dia_ref  = np.genfromtxt("ivus_rest/diastolic_reference_points.csv", delimiter='\t')
sys_cont = np.genfromtxt("ivus_rest/systolic_contours.csv",          delimiter='\t')
sys_ref  = np.genfromtxt("ivus_rest/systolic_reference_points.csv",  delimiter='\t')

dia_input = mm.numpy_to_inputdata(
    lumen_arr=dia_cont, reference_points=dia_ref,
    record=record, diastole=True, label="diastole_rest",
)
sys_input = mm.numpy_to_inputdata(
    lumen_arr=sys_cont, reference_points=sys_ref,
    record=record, diastole=False, label="systole_rest",
)

rest_array, (dia_logs, sys_logs) = mm.from_array_singlepair(
    input_data_a=dia_input,
    input_data_b=sys_input,
    step_rotation_deg=0.01,
    range_rotation_deg=60,
    output_path="output/rest_array",
    interpolation_steps=0,
    smooth=True,
    postprocessing=True,
)
print("Logs (dia):", dia_logs[:3], "...")


✅ Successfully built geometry from input data
-----------------------------------------
✅ Lumen
❌ Eem
✅ Catheter
❌ Structures (0)
-----------------------------------------
Label: diastole_rest
Diastole phase: Yes

Logs (dia): [(1, 0, -7.709999999999979, -0.12124275208313184, 0.3603055547533245, 3.5949458260233564, 5.610358740916264), (2, 1, -43.34, -1.0674125981869906, 2.7305771329569084, 3.5949458260233564, 5.610358740916264), (3, 2, 11.049999999999999, -0.6964904200670023, 2.598353951083425, 3.5949458260233564, 5.610358740916264)] ...

✅ Successfully built geometry from input data
-----------------------------------------
✅ Lumen
❌ Eem
✅ Catheter
❌ Structures (0)
-----------------------------------------
Label: systole_rest
Diastole phase: No


+-------------------------------------------------------------------+
|       ✅ Finished aligning 'diastole_rest' (anomalous: true)       |
+---------+------------+---------------+-------+------+-------------+
| Contour | Matched To | Rotatio

## 4. Alignment with a centerline

A `PyCenterline` is built from a raw (N×3) coordinate array — no index column required:

```python
cl_raw    = np.genfromtxt("centerline_raw.csv", delimiter=',')
centerline = mm.numpy_to_centerline(cl_raw)
```

`align_three_point` co-registers a geometry pair onto the centerline using three anatomical
landmarks: an aortic reference point, an upper vessel point, and a lower vessel point.
The preferred method is `align_combined`, which additionally optimizes against a CCTA point cloud
(see the [CCTA tutorial](../tutorial_ccta.rst) for how to prepare `results["rca_points"]`).

In [13]:
cl_raw = np.genfromtxt("centerline_raw.csv", delimiter=',')
cl     = mm.numpy_to_centerline(cl_raw)

aligned_geometry, spacing_mm, total_rotation_deg = mm.align_three_point(
    centerline=cl,
    geometry=rest,   # PyGeometryPair from from_file_full above
    main_ref_pt=(12.2605, -201.3643, 1751.0554),
    counterclockwise_ref_pt=(11.7567, -202.1920, 1754.7975),
    clockwise_ref_pt=(15.6605, -202.1920, 1749.9655),
    write=True,
    watertight=False,
    interpolation_steps=0,
)
# Resample `cl` to the same spacing align_three_point derived internally
# (from the mean spacing between consecutive contour centroids in `rest`),
# instead of re-deriving it.
resampled_cl = cl.resample(spacing_mm)
print("Resampled centerline (first 3 points):", mm.to_array(resampled_cl)[:3])

---------------------Centerline alignment: Finding optimal rotation---------------------
Resampled centerline (first 3 points): [[ 0.00000000e+00  1.30847000e+01 -2.00350800e+02  1.75186020e+03]
 [ 1.00000000e+00  1.35728283e+01 -2.01439230e+02  1.75228207e+03]
 [ 2.00000000e+00  1.38230940e+01 -2.02663633e+02  1.75244697e+03]]
✅ Best angle found: 73.00°

Saving files for 'None' to 'output/aligned'
LUMEN .obj files: 2/2 written successfully
CATHETER .obj files: 2/2 written successfully
WALL .obj files: 2/2 written successfully


In [14]:
# Write before/after alignment meshes with explicit names and visualise
mm.to_obj(aligned_geometry.geom_a, "output/aligned", watertight=False,
          contour_types=[mm.PyContourType.Lumen], filename_prefix="dia")
mm.to_obj(aligned_geometry.geom_b, "output/aligned", watertight=False,
          contour_types=[mm.PyContourType.Lumen], filename_prefix="sys")

plot_pair(
    before_paths=["output/after/dia_lumen.obj", "output/after/sys_lumen.obj"],
    after_paths=["output/aligned/dia_lumen.obj", "output/aligned/sys_lumen.obj"],
    colors=["royalblue", "firebrick"],
    titles=["Before Alignment", "After Alignment"],
)

Successfully wrote lumen to output/aligned/dia_lumen.obj
Successfully wrote lumen to output/aligned/sys_lumen.obj


## 5. Saving geometries as `.obj` files

`to_obj` can be called on any `PyGeometryPair` or `PyGeometry` to export all or selected contour
layers. The `filename_prefix` determines the output file stem, and `contour_types` selects which
layers to export:

In [15]:
# Export lumen and catheter from the aligned rest geometry
mm.to_obj(
    rest.geom_a,
    "output/rest_export",
    watertight=False,
    contour_types=[mm.PyContourType.Lumen, mm.PyContourType.Catheter],
    filename_prefix="aligned",
)
# Creates: output/rest_export/aligned_lumen.obj  and  aligned_catheter.obj
import os
print(os.listdir("output/rest_export"))

Successfully wrote lumen to output/rest_export/aligned_lumen.obj
Successfully wrote catheter to output/rest_export/aligned_catheter.obj
['aligned_catheter.mtl', 'aligned_catheter.obj', 'aligned_lumen.mtl', 'aligned_lumen.obj']


## 6. Utility functions: `to_array` and `numpy_to_geometry`

### `to_array`

`to_array` is a single-dispatch converter from any `Py*` object to numpy arrays.
The return type depends on the input:

| Input type | Return |
|------------|--------|
| `PyContour` or `PyCenterline` | `ndarray` shape `(N, 4)`: frame_index, x, y, z |
| `PyFrame` or `PyGeometry` | `dict[str, ndarray]` with keys `lumen`, `eem`, … `reference` |
| `PyGeometryPair` | `tuple[dict, dict]` — one dict per geometry |
| `PyInputData` | `dict` with layer keys plus `diastole` and `label` |

In [16]:
# PyGeometryPair → (dict, dict)
rest_dia_arr, rest_sys_arr = mm.to_array(rest)
print("Lumen array shape (geom_a):", rest_dia_arr["lumen"].shape)

# PyGeometry → dict
geom_arr = mm.to_array(rest.geom_a)
print("Keys:", list(geom_arr.keys()))

# PyCenterline → ndarray
cl_arr = mm.to_array(resampled_cl)
print("Centerline array shape:", cl_arr.shape)

# PyContour → ndarray
contour_arr = mm.to_array(rest.geom_a.frames[-1].lumen)
print("Contour array shape:", contour_arr.shape)

# PyInputData → dict
input_arr = mm.to_array(dia_input)
print("InputData keys:", list(input_arr.keys()))

Lumen array shape (geom_a): (10020, 4)
Keys: ['lumen', 'eem', 'catheter', 'wall', 'reference', 'structures']
Centerline array shape: (129, 4)
Contour array shape: (501, 4)
InputData keys: ['lumen', 'eem', 'reference', 'diastole', 'label', 'records', 'structures']


### `numpy_to_geometry`

When you already have aligned contour data as numpy arrays and want to construct a `PyGeometry`
directly — for example after custom post-processing — use `numpy_to_geometry`:

In [17]:
# Reconstruct a PyGeometry from numpy arrays
lumen_np = rest_dia_arr["lumen"]
ref_np   = rest_dia_arr["reference"]

reconstructed = mm.numpy_to_geometry(
    lumen_arr=lumen_np,
    eem_arr=np.array([]),
    catheter_arr=np.array([]),
    wall_arr=np.array([]),
    reference_arr=ref_np,
    label="reconstructed_rest_dia",
)
print(f"Frames: {len(reconstructed.frames)}, "
      f"first frame lumen points: {len(reconstructed.frames[0].lumen.points)}")

Frames: 20, first frame lumen points: 501


/mnt/d/00_coding/multimoda-rs/multimodars/_converters.py:685: UserWarning: dropped reference point(s) on frame(s) [385] without lumen rows
  warnings.warn(


## 7. Class-level methods

### `PyContour`

All transformation methods return a **new** object and leave the original unchanged:

In [18]:
contour = rest.geom_a.frames[0].lumen

# Geometry queries
area          = contour.get_area()
elliptic_ratio = contour.get_elliptic_ratio()
pts_list      = contour.points_as_tuples()
(p1, p2), dist_close  = contour.find_closest_opposite()
(p1, p2), dist_far    = contour.find_farthest_points()

print(f"Area: {area:.4f} mm²  |  Elliptic ratio: {elliptic_ratio:.4f}")
print(f"Min diameter: {dist_close:.4f} mm  |  Max diameter: {dist_far:.4f} mm")

# Transformations (all return new PyContour)
contour_rot  = contour.rotate(20.0)
contour_trsl = contour_rot.translate(0.0, 1.0, 2.0)
contour_sort = contour.sort_contour_points()

# Write a modified contour back into the geometry
old_frame = rest.geom_a.frames[2]
new_lumen = old_frame.lumen.rotate(20.0)
new_frame = mm.PyFrame(
    id=old_frame.id,
    centroid=old_frame.centroid,
    lumen=new_lumen,
    extras=old_frame.extras,
    reference_point=old_frame.reference_point,
)
modified_geom = rest.geom_a.replace_frame(2, new_frame)
print("Frame 2 replaced:", modified_geom.frames[2].lumen is new_lumen)

Area: 5.6366 mm²  |  Elliptic ratio: 4.5865
Min diameter: 1.1578 mm  |  Max diameter: 5.3083 mm
Frame 2 replaced: False


### `PyFrame` and `PyGeometry`

`PyFrame` mirrors the contour transformations and applies them to all layers:

In [19]:
frame = rest.geom_a.frames[0]
frame_rot    = frame.rotate(20.0)
frame_trsl   = frame.translate(0.0, 1.0, 2.0)
frame_sorted = frame.sort_frame_points()

# PyGeometry geometry-level operations
geom_smooth  = rest.geom_a.smooth_frames()
geom_ds      = rest.geom_a.downsample(100)
geom_sorted  = rest.geom_a.sort_frame_points()
geom_centred = rest.geom_a.center_to_contour(mm.PyContourType.Catheter)
geom_rot     = rest.geom_a.rotate(20.0)
geom_trsl    = geom_rot.translate(0.0, 1.0, 2.0)

print(f"Original contour points: {len(rest.geom_a.frames[0].lumen.points)}")
print(f"Downsampled:             {len(geom_ds.frames[0].lumen.points)}")

# Frame access
frame_by_idx = rest.geom_a.get_frame_at_index(5)
frame_by_z   = rest.geom_a.get_frame_at_z(12.5)
lumen_list   = rest.geom_a.get_lumen_contours()
wall_list    = rest.geom_a.get_contours_by_type("Wall")

print(f"Frame at index 5 z: {frame_by_idx.lumen.points[0].z:.3f}")
print(f"Total lumen contours: {len(lumen_list)}")

# PyGeometryPair summary
(summary_a, summary_b), deform = rest.get_summary()
print(f"\nPair summary A: {summary_a}")
print(f"Pair summary B: {summary_b}")
print(f"Deformation table shape: {np.array(deform).shape}")

Original contour points: 501
Downsampled:             100
Frame at index 5 z: 6.351
Total lumen contours: 20
+----+----------+-----------+----------+-----------+-------+
| id | area_dia | ellip_dia | area_sys | ellip_sys |   z   |
+----+----------+-----------+----------+-----------+-------+
| 0  | 5.64     | 4.59      | 5.66     | 4.94      | 0.00  |
| 1  | 5.80     | 4.75      | 5.59     | 5.08      | 1.27  |
| 2  | 5.70     | 3.70      | 5.62     | 4.87      | 2.54  |
| 3  | 5.56     | 2.57      | 6.05     | 4.40      | 3.81  |
| 4  | 5.59     | 1.64      | 6.25     | 3.48      | 5.08  |
| 5  | 6.42     | 1.51      | 6.50     | 2.85      | 6.35  |
| 6  | 7.26     | 1.37      | 6.84     | 2.20      | 7.62  |
| 7  | 7.74     | 1.25      | 7.52     | 1.93      | 8.89  |
| 8  | 7.77     | 1.43      | 7.59     | 1.69      | 10.16 |
| 9  | 9.05     | 1.33      | 7.35     | 1.63      | 11.43 |
| 10 | 11.92    | 1.18      | 9.70     | 1.37      | 12.70 |
| 11 | 15.27    | 1.08      | 11.51  